# Calibración secuencial del Algoritmo Genético

Notebook para la calibracion sobre las 3 instancias
(`small`, `medium`, `large`). Se ejecuta **una fase por vez**, en este orden:

1. **E01_PM** - probabilidad de mutacion (PM) sola
2. **E02_PC** - probabilidad de cruce (PC), con PM ya fijada
3. **E03_PMPC** - PM y PC de forma conjunta (rejilla combinada)
4. **E04_N** - tamano de poblacion (N) solo
5. **E05_G** - numero de generaciones (G) solo
6. **E06_NG** - N y G de forma conjunta
7. **E07_TORNEO** - toneo
8. **E08_INIT** - inicialización
9. **E09_MREF** - modo referencia Nivel 1
10. **E10_PRESUP** - presupuesto computacional
11. **E11_PARADA** - criterio de parada

**Flujo de cada fase (igual para todas):**
1. Se ejecuta para las 3 instancias, con 20 semillas cada una
   (`config_experimentos.SEMILLAS`).
2. Se leen `resumen.csv` y el libro `<exp_id>_por_instancia.xlsx` (una hoja
   por instancia, reescrita si ya existia).
3. Se muestra, por instancia, la configuracion ganadora: **mayor tasa de
   factibilidad -> menor LCOH medio -> menor desviacion tipica -> menor
   tiempo** (en ese orden lexicografico).
4. **EL USUARIO decide** el valor que se fija.
5. Se escribe la decision en `config_experimentos.CALIBRADO` para que las
   fases siguientes la hereden automaticamente.
6. Se generan las figuras y la tabla LaTeX de la fase.


In [ ]:
# =====================================================================
# CELDA 1 - SETUP: montar Drive, localizar EXPERIMENTOS/ e importar
# =====================================================================
import os, sys, time

# --- Montar Google Drive --------------------------
DRIVE_OK = False
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_OK = True
except Exception as _e:
    print("[INFO] No se pudo montar Drive (fuera de Colab?):", _e)

# --- Localizar la carpeta EXPERIMENTOS/ -----
DIR_NOTEBOOK = None
raiz_busqueda = "/content/drive" if os.path.isdir("/content/drive") else os.getcwd()
candidatos = []
for raiz, _dirs, ficheros in os.walk(raiz_busqueda):
    if "_rutas.py" in ficheros:
        candidatos.append(raiz)
candidatos.sort()
if candidatos:
    DIR_NOTEBOOK = candidatos[0]
    print("Carpeta EXPERIMENTOS encontrada:", DIR_NOTEBOOK)
    if len(candidatos) > 1:
        print("  [AVISO] Hay varias carpetas con _rutas.py, se usa la primera:", candidatos)

    # Estructura esperada: .../TFM/{EXPERIMENTOS, CODIGO_FUENTE, DATOS/tablas}
    # (EXPERIMENTOS y CODIGO_FUENTE son carpetas HERMANAS bajo TFM/; las
    # instancias JSON viven en TFM/DATOS/tablas).
    DIR_TFM_DETECTADO = os.path.dirname(DIR_NOTEBOOK)
    print("Carpeta TFM detectada         :", DIR_TFM_DETECTADO)
    for sub in ("CODIGO_FUENTE", os.path.join("DATOS", "tablas")):
        ruta_sub = os.path.join(DIR_TFM_DETECTADO, sub)
        print(f"  {sub:<14} {'OK' if os.path.isdir(ruta_sub) else '[AVISO] no encontrada'} -> {ruta_sub}")
else:
    raise FileNotFoundError(
        "No se encontro ninguna carpeta con '_rutas.py' dentro de " + raiz_busqueda +
        ". Comprueba que EXPERIMENTOS/ esta subida a tu Drive y que el "
        "montaje de Drive se completo correctamente."
    )

if DIR_NOTEBOOK not in sys.path:
    sys.path.insert(0, DIR_NOTEBOOK)

import _rutas
import agregacion
import calibracion
import config_experimentos as C
import exportar
import exportar_calibracion
import graficas
import instancias as ins

print("DIR_RAIZ       :", _rutas.DIR_RAIZ)
print("DIR_RESULTADOS :", _rutas.DIR_RESULTADOS)
print("Instancias     :", C.INSTANCIAS)
print("N semillas     :", len(C.SEMILLAS), "->", C.SEMILLAS)
print()
print("CALIBRADO vigente:")
for k, v in C.CALIBRADO.items():
    print(f"  {k:<16} {'(sin fijar)' if v is None else v}")


Mounted at /content/drive
Carpeta EXPERIMENTOS encontrada: /content/drive/MyDrive/TFM/EXPERIMENTOS
Carpeta TFM detectada         : /content/drive/MyDrive/TFM
  CODIGO_FUENTE  OK -> /content/drive/MyDrive/TFM/CODIGO_FUENTE
  DATOS/tablas   OK -> /content/drive/MyDrive/TFM/DATOS/tablas
DIR_RAIZ       : /content/drive/MyDrive/TFM/CODIGO_FUENTE
DIR_RESULTADOS : /content/drive/MyDrive/TFM/EXPERIMENTOS/RESULTADOS
Instancias     : ['small', 'medium', 'large']
N semillas     : 20 -> [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]

CALIBRADO vigente:
  prob_mutacion    (sin fijar)
  prob_cruce       (sin fijar)
  tam_poblacion    (sin fijar)
  n_generaciones   (sin fijar)
  k_torneo         (sin fijar)
  tipo_init        (sin fijar)
  frac_semilla     (sin fijar)
  m_ref            (sin fijar)


In [ ]:
# =====================================================================
# CELDA 2 - UTILIDADES: resumen de una fase
# =====================================================================
import pandas as pd


def resumen_fase(exp_id, r):
    """Muestra, por instancia, la configuracion ganadora segun el criterio
    (1) tasa_factibilidad, (2) lcoh_media, (3) lcoh_std, (4) tiempo_medio_s,
    y el tiempo total/medio real de la campana."""
    res = r["resumen"]
    runs = r["runs"]
    cols_rejilla = r["cols_rejilla"]

    print("=" * 78)
    print(f"RESUMEN DE {exp_id}")
    print("=" * 78)

    t_total = pd.to_numeric(runs["tiempo_s"], errors="coerce").fillna(0.0).sum()
    t_medio = pd.to_numeric(runs["tiempo_s"], errors="coerce").mean()
    print(f"Tiempo total real: {t_total:.1f} s ({t_total/60:.1f} min)  |  "
          f"Tiempo medio/ejecucion: {t_medio:.2f} s\n")

    ganadoras = {}
    for inst, sub in res.groupby("instancia", sort=False):
        g = sub[sub["es_mejor_en_instancia"] == True]
        if g.empty:
            print(f"  [{inst}] sin ninguna configuracion con datos.")
            continue
        fila = g.iloc[0]
        ganadoras[inst] = fila["id_config"]
        params = {c: fila[c] for c in cols_rejilla if c in fila.index}
        print(f"  [{inst:<8}] ganador: {fila['id_config']:<28} params={params}")
        print(f"             LCOH medio = {fila['lcoh_media']:.4f}  "
              f"std = {fila['lcoh_std']:.4f}  "
              f"factibilidad = {fila['tasa_factibilidad']:.2f}  "
              f"t medio = {fila['tiempo_medio_s']:.2f}s")

    if r.get("ranking") is not None and not r["ranking"].empty:
        print("\n  Ranking agregado (mas veces mejor entre instancias):")
        top = r["ranking"].iloc[0]
        print(f"    {top['id_config']}  ->  mejor en {top['veces_mejor_txt']} instancias")

    print("=" * 78)
    return ganadoras


---
## FASE 1 - E01_PM: probabilidad de mutacion

Rejilla: `prob_mutacion in [0.001, 0.01, 0.05, 0.1, 0.2]`, con PC=1.0,
N=40, G=100 fijos (valores de arranque, `BASE` en `config_experimentos.py`).


In [ ]:
# =====================================================================
# E01_PM - EJECUCION
# =====================================================================
r_E01 = calibracion.ejecutar_calibracion(
    exp_id="E01_PM",
    instancias_ids=C.INSTANCIAS,
    semillas=C.SEMILLAS,
    guardar_convergencia=True,
    con_ranking=True,
    verbose=True,
)

graficas.generar_todas(
    exp_id="E01_PM", res=r_E01["resumen"], runs=r_E01["runs"],
    conv=pd.read_csv(r_E01["ruta_convergencia"]) if os.path.isfile(r_E01["ruta_convergencia"]) else None,
    cols_rejilla=r_E01["cols_rejilla"], nombre_exp=r_E01["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E01_PM", runs=r_E01["runs"], res=r_E01["resumen"],
    ranking=r_E01.get("ranking"), nombre_exp=r_E01["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)

# Libro Excel "por instancia" de ESTA calibracion (E01_PM_por_instancia.xlsx):
for inst_etq in list(dict.fromkeys(r_E01["resumen"]["instancia"].tolist())):
    sub = r_E01["resumen"][r_E01["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E01["runs"].loc[r_E01["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E01["runs"].loc[r_E01["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E01_PM", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E01["cols_rejilla"], nombre_exp=r_E01["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E01_PM  |  Calibracion de la probabilidad de mutacion
  Instancias      : small, medium, large
  Configuraciones : 5  (PM=0.001, PM=0.01, PM=0.05, PM=0.1, PM=0.2)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 300
  Parametros fijos: N=40, G=100, PC=1.0, PM=None, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   PM=0.001                     fact=5/20  LCOH medio=5.0045          [  6.7%]
   PM=0.01                      fact=6/20  LCOH medio=4.9175          [ 13.3%]
   PM=0.05                      fact=8/20  LCOH medio=4.7982          [ 20.0%]
   PM=0.1                       fact=14/20  LCOH medio=4.7104          [ 26.7%]
   PM=0.2                       fact=11/20  LCOH medio=4.6764          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   PM=0.001                     fact=4/20  LCOH medio=5.2281          [ 40.0%]
   PM=0.01                      fact=7/

In [ ]:
# =====================================================================
# E01_PM - RESUMEN Y DECISION
# =====================================================================
ganadoras_E01 = resumen_fase("E01_PM", r_E01)

VALOR_PM_ELEGIDO = 0.1   # Editamos este dato para fijar el PM ganador

if VALOR_PM_ELEGIDO is not None:
    C.CALIBRADO["prob_mutacion"] = VALOR_PM_ELEGIDO
    print(f"CALIBRADO['prob_mutacion'] = {VALOR_PM_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_PM_ELEGIDO. Las fases siguientes "
          "usaran BASE['prob_mutacion'] hasta que lo hagas.")


CALIBRADO['prob_mutacion'] = 0.1  (fijado)


---
## FASE 2 - E02_PC: probabilidad de cruce (con PM ya fijada)

Rejilla: `prob_cruce in [0.6, 0.75, 0.9, 1.0]`. Hereda `prob_mutacion` de
`CALIBRADO` (fijado en la fase anterior) automaticamente a traves de
`config_base_efectiva`.


In [ ]:
# =====================================================================
# E02_PC - EJECUCION
# =====================================================================
r_E02 = calibracion.ejecutar_calibracion(
    exp_id="E02_PC", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E02_PC", res=r_E02["resumen"], runs=r_E02["runs"],
    conv=pd.read_csv(r_E02["ruta_convergencia"]) if os.path.isfile(r_E02["ruta_convergencia"]) else None,
    cols_rejilla=r_E02["cols_rejilla"], nombre_exp=r_E02["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E02_PC", runs=r_E02["runs"], res=r_E02["resumen"],
    ranking=r_E02.get("ranking"), nombre_exp=r_E02["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E02["resumen"]["instancia"].tolist())):
    sub = r_E02["resumen"][r_E02["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E02["runs"].loc[r_E02["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E02["runs"].loc[r_E02["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E02_PC", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E02["cols_rejilla"], nombre_exp=r_E02["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E02_PC  |  Calibracion de la probabilidad de cruce
  Instancias      : small, medium, large
  Configuraciones : 4  (PC=0.6, PC=0.75, PC=0.9, PC=1)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 240
  Parametros fijos: N=40, G=100, PC=1.0, PM=0.1, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   PC=0.6                       fact=14/20  LCOH medio=4.6480          [  8.3%]
   PC=0.75                      fact=12/20  LCOH medio=4.6520          [ 16.7%]
   PC=0.9                       fact=13/20  LCOH medio=4.6940          [ 25.0%]
   PC=1                         fact=14/20  LCOH medio=4.7104          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   PC=0.6                       fact=19/20  LCOH medio=4.7930          [ 41.7%]
   PC=0.75                      fact=18/20  LCOH medio=4.8470          [ 50.0%]
   PC=0.9                       fact=19/20  LCOH med

In [ ]:
ganadoras_E02 = resumen_fase("E02_PC", r_E02)

VALOR_PC_ELEGIDO = 0.6   # <-- EDITAMOS ESTO
if VALOR_PC_ELEGIDO is not None:
    C.CALIBRADO["prob_cruce"] = VALOR_PC_ELEGIDO
    print(f"CALIBRADO['prob_cruce'] = {VALOR_PC_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_PC_ELEGIDO.")


CALIBRADO['prob_cruce'] = 0.6  (fijado)


---
## FASE 3 - E03_PMPC: PM y PC de forma conjunta

Rejilla combinada (producto cartesiano) alrededor de los valores ganadores
de las fases 1 y 2, para comprobar si existe interacción entre ambos
parámetros que la calibracion secuencial (uno a uno) no detecta.


In [ ]:
# =====================================================================
# E03_PMPC - EJECUCION
# =====================================================================
r_E03 = calibracion.ejecutar_calibracion(
    exp_id="E03_PMPC", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E03_PMPC", res=r_E03["resumen"], runs=r_E03["runs"],
    conv=pd.read_csv(r_E03["ruta_convergencia"]) if os.path.isfile(r_E03["ruta_convergencia"]) else None,
    cols_rejilla=r_E03["cols_rejilla"], nombre_exp=r_E03["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E03_PMPC", runs=r_E03["runs"], res=r_E03["resumen"],
    ranking=r_E03.get("ranking"), nombre_exp=r_E03["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E03["resumen"]["instancia"].tolist())):
    sub = r_E03["resumen"][r_E03["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E03["runs"].loc[r_E03["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E03["runs"].loc[r_E03["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E03_PMPC", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E03["cols_rejilla"], nombre_exp=r_E03["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E03_PMPC  |  Calibracion combinada de PM y PC
  Instancias      : small, medium, large
  Configuraciones : 9  (PM=0.05_PC=0.6, PM=0.05_PC=0.8, PM=0.05_PC=1, PM=0.1_PC=0.6, PM=0.1_PC=0.8, PM=0.1_PC=1 ...)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 540
  Parametros fijos: N=40, G=100, PC=0.6, PM=None, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   PM=0.05_PC=0.6               fact=10/20  LCOH medio=4.6762          [  3.7%]
   PM=0.05_PC=0.8               fact=9/20  LCOH medio=4.6945          [  7.4%]
   PM=0.05_PC=1                 fact=8/20  LCOH medio=4.7982          [ 11.1%]
   PM=0.1_PC=0.6                fact=14/20  LCOH medio=4.6480          [ 14.8%]
   PM=0.1_PC=0.8                fact=10/20  LCOH medio=4.6468          [ 18.5%]
   PM=0.1_PC=1                  fact=14/20  LCOH medio=4.7104          [ 22.2%]
   PM=0.2_PC=0.6                fact=12/20  LCO

In [ ]:
ganadoras_E03 = resumen_fase("E03_PMPC", r_E03)

# Si la combinacion conjunta mejora sobre la secuencial, actualizamos las nuevas calibraciones
# prob_mutacion y prob_cruce con los valores conjuntos ganadores.
VALOR_PM_CONJUNTO = 0.2   # <-- EDITAMOS SI ES EL CASO, SINO LO DEJAMOS EN NONE
VALOR_PC_CONJUNTO = 0.6   # <-- EDITA SI ES EL CASO, SINO LO DEJAMOS EN NONE

if VALOR_PM_CONJUNTO is not None:
    C.CALIBRADO["prob_mutacion"] = VALOR_PM_CONJUNTO
    print(f"CALIBRADO['prob_mutacion'] = {VALOR_PM_CONJUNTO}  (actualizado tras E03)")
if VALOR_PC_CONJUNTO is not None:
    C.CALIBRADO["prob_cruce"] = VALOR_PC_CONJUNTO
    print(f"CALIBRADO['prob_cruce'] = {VALOR_PC_CONJUNTO}  (actualizado tras E03)")
if VALOR_PM_CONJUNTO is None and VALOR_PC_CONJUNTO is None:
    print("Se mantienen los valores de PM y PC fijados en E01/E02.")


CALIBRADO['prob_mutacion'] = 0.2  (actualizado tras E03)
CALIBRADO['prob_cruce'] = 0.6  (actualizado tras E03)


---
## FASE 4 - E04_N: tamano de la poblacion

Rejilla: `tam_poblacion in [20, 40, 80, 150]`, con PM y PC ya fijados
(heredados de las fases anteriores via `CALIBRADO`).


In [ ]:
# =====================================================================
# E04_N - EJECUCION
# =====================================================================
r_E04 = calibracion.ejecutar_calibracion(
    exp_id="E04_N", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E04_N", res=r_E04["resumen"], runs=r_E04["runs"],
    conv=pd.read_csv(r_E04["ruta_convergencia"]) if os.path.isfile(r_E04["ruta_convergencia"]) else None,
    cols_rejilla=r_E04["cols_rejilla"], nombre_exp=r_E04["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E04_N", runs=r_E04["runs"], res=r_E04["resumen"],
    ranking=r_E04.get("ranking"), nombre_exp=r_E04["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E04["resumen"]["instancia"].tolist())):
    sub = r_E04["resumen"][r_E04["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E04["runs"].loc[r_E04["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E04["runs"].loc[r_E04["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E04_N", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E04["cols_rejilla"], nombre_exp=r_E04["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E04_N  |  Calibracion del tamano de la poblacion
  Instancias      : small, medium, large
  Configuraciones : 4  (N=50, N=100, N=150, N=200)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 240
  Parametros fijos: N=40, G=100, PC=0.6, PM=0.2, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   N=50                         fact=14/20  LCOH medio=4.6160          [  8.3%]
   N=100                        fact=19/20  LCOH medio=4.6107          [ 16.7%]
   N=150                        fact=20/20  LCOH medio=4.6078          [ 25.0%]
   N=200                        fact=20/20  LCOH medio=4.6034          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   N=50                         fact=20/20  LCOH medio=4.6959          [ 41.7%]
   N=100                        fact=20/20  LCOH medio=4.6334          [ 50.0%]
   N=150                        fact=20/20  LCOH medio=4.6

In [ ]:
ganadoras_E04 = resumen_fase("E04_N", r_E04)

VALOR_N_ELEGIDO = 150   # <-- EDITAMOS ESTO

if VALOR_N_ELEGIDO is not None:
    C.CALIBRADO["tam_poblacion"] = VALOR_N_ELEGIDO
    print(f"CALIBRADO['tam_poblacion'] = {VALOR_N_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_N_ELEGIDO.")


CALIBRADO['tam_poblacion'] = 150  (fijado)


---
## FASE 5 - E05_G: numero de generaciones (con N ya fijado)

Rejilla: `n_generaciones in [50, 100, 200, 400]`.


In [ ]:
# =====================================================================
# E05_G - EJECUCION
# =====================================================================
r_E05 = calibracion.ejecutar_calibracion(
    exp_id="E05_G", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E05_G", res=r_E05["resumen"], runs=r_E05["runs"],
    conv=pd.read_csv(r_E05["ruta_convergencia"]) if os.path.isfile(r_E05["ruta_convergencia"]) else None,
    cols_rejilla=r_E05["cols_rejilla"], nombre_exp=r_E05["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E05_G", runs=r_E05["runs"], res=r_E05["resumen"],
    ranking=r_E05.get("ranking"), nombre_exp=r_E05["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E05["resumen"]["instancia"].tolist())):
    sub = r_E05["resumen"][r_E05["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E05["runs"].loc[r_E05["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E05["runs"].loc[r_E05["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E05_G", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E05["cols_rejilla"], nombre_exp=r_E05["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E05_G  |  Calibracion del numero de generaciones
  Instancias      : small, medium, large
  Configuraciones : 4  (G=50, G=100, G=200, G=400)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 240
  Parametros fijos: N=150, G=100, PC=0.6, PM=0.2, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   G=50                         fact=20/20  LCOH medio=4.6109          [  8.3%]
   G=100                        fact=20/20  LCOH medio=4.6078          [ 16.7%]
   G=200                        fact=20/20  LCOH medio=4.6078          [ 25.0%]
   G=400                        fact=20/20  LCOH medio=4.6078          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   G=50                         fact=20/20  LCOH medio=4.7348          [ 41.7%]
   G=100                        fact=20/20  LCOH medio=4.6022          [ 50.0%]
   G=200                        fact=20/20  LCOH medio=4.

In [ ]:
ganadoras_E05 = resumen_fase("E05_G", r_E05)

VALOR_G_ELEGIDO = 200   # <-- EDITA ESTO, p.ej. 200

if VALOR_G_ELEGIDO is not None:
    C.CALIBRADO["n_generaciones"] = VALOR_G_ELEGIDO
    print(f"CALIBRADO['n_generaciones'] = {VALOR_G_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_G_ELEGIDO.")


CALIBRADO['n_generaciones'] = 200  (fijado)


---
## FASE 6 - E06_NG: N y G de forma conjunta

Rejilla combinada alrededor de los valores ganadores de las fases 4 y 5,
para comprobar interaccion entre tamano de poblacion y numero de
generaciones (p. ej. si una poblacion mayor permite reducir generaciones
sin perder calidad).


In [ ]:
# =====================================================================
# E06_NG - EJECUCION
# =====================================================================
r_E06 = calibracion.ejecutar_calibracion(
    exp_id="E06_NG", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E06_NG", res=r_E06["resumen"], runs=r_E06["runs"],
    conv=pd.read_csv(r_E06["ruta_convergencia"]) if os.path.isfile(r_E06["ruta_convergencia"]) else None,
    cols_rejilla=r_E06["cols_rejilla"], nombre_exp=r_E06["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E06_NG", runs=r_E06["runs"], res=r_E06["resumen"],
    ranking=r_E06.get("ranking"), nombre_exp=r_E06["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E06["resumen"]["instancia"].tolist())):
    sub = r_E06["resumen"][r_E06["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E06["runs"].loc[r_E06["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E06["runs"].loc[r_E06["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E06_NG", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E06["cols_rejilla"], nombre_exp=r_E06["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )


E06_NG  |  Calibracion combinada de tamano de poblacion y generaciones
  Instancias      : small, medium, large
  Configuraciones : 9  (N=20_G=50, N=20_G=100, N=20_G=200, N=40_G=50, N=40_G=100, N=40_G=200 ...)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 540
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   N=20_G=50                    fact=11/20  LCOH medio=4.7136          [  3.7%]
   N=20_G=100                   fact=11/20  LCOH medio=4.6814          [  7.4%]
   N=20_G=200                   fact=11/20  LCOH medio=4.6555          [ 11.1%]
   N=40_G=50                    fact=12/20  LCOH medio=4.6828          [ 14.8%]
   N=40_G=100                   fact=12/20  LCOH medio=4.6348          [ 18.5%]
   N=40_G=200                   fact=12/20  LCOH medio=4.6218          [ 22.2%]
   N=80_G=50                    fact=15

In [ ]:
ganadoras_E06 = resumen_fase("E06_NG", r_E06)

VALOR_N_CONJUNTO = None   # <-- EDITA (o deja None para mantener lo de E04)
VALOR_G_CONJUNTO = None   # <-- EDITA (o deja None para mantener lo de E05)

if VALOR_N_CONJUNTO is not None:
    C.CALIBRADO["tam_poblacion"] = VALOR_N_CONJUNTO
    print(f"CALIBRADO['tam_poblacion'] = {VALOR_N_CONJUNTO}  (actualizado tras E06)")
if VALOR_G_CONJUNTO is not None:
    C.CALIBRADO["n_generaciones"] = VALOR_G_CONJUNTO
    print(f"CALIBRADO['n_generaciones'] = {VALOR_G_CONJUNTO}  (actualizado tras E06)")
if VALOR_N_CONJUNTO is None and VALOR_G_CONJUNTO is None:
    print("Se mantienen los valores de N y G fijados en E04/E05.")


Se mantienen los valores de N y G fijados en E04/E05.


---
## FASE 7 - E07_TORNEO: presión selectiva (tamaño del torneo)

Rejilla: `k_torneo in [2, 3, 5, 7]`, con $p_m=0.2$, $p_c=0.6$, $N=150$,
$G=200$ ya fijados (heredados automáticamente de las fases anteriores a
través de `config_base_efectiva`).

In [ ]:
# =====================================================================
# E07_TORNEO - EJECUCION
# =====================================================================
r_E07 = calibracion.ejecutar_calibracion(
    exp_id="E07_TORNEO", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E07_TORNEO", res=r_E07["resumen"], runs=r_E07["runs"],
    conv=pd.read_csv(r_E07["ruta_convergencia"]) if os.path.isfile(r_E07["ruta_convergencia"]) else None,
    cols_rejilla=r_E07["cols_rejilla"], nombre_exp=r_E07["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E07_TORNEO", runs=r_E07["runs"], res=r_E07["resumen"],
    ranking=r_E07.get("ranking"), nombre_exp=r_E07["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E07["resumen"]["instancia"].tolist())):
    sub = r_E07["resumen"][r_E07["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E07["runs"].loc[r_E07["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E07["runs"].loc[r_E07["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E07_TORNEO", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E07["cols_rejilla"], nombre_exp=r_E07["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

E07_TORNEO  |  Calibracion de la presion selectiva (tamano del torneo)
  Instancias      : small, medium, large
  Configuraciones : 3  (k=2, k=3, k=5)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 180
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=3, init=A

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   k=2                          fact=20/20  LCOH medio=4.6016          [ 11.1%]
   k=3                          fact=20/20  LCOH medio=4.6078          [ 22.2%]
   k=5                          fact=20/20  LCOH medio=4.6039          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   k=2                          fact=20/20  LCOH medio=4.5354          [ 44.4%]
   k=3                          fact=20/20  LCOH medio=4.5442          [ 55.6%]
   k=5                          fact=20/20  LCOH medio=4.5265          [ 66.7%]

[large]  |P|=12 |J|=30 |K|=55 HTotal=53000 kg/

In [ ]:
ganadoras_E07 = resumen_fase("E07_TORNEO", r_E07)

VALOR_K_ELEGIDO = 2   # <-- EDITA ESTO, p.ej. 5

if VALOR_K_ELEGIDO is not None:
    C.CALIBRADO["k_torneo"] = VALOR_K_ELEGIDO
    print(f"CALIBRADO['k_torneo'] = {VALOR_K_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_K_ELEGIDO.")

CALIBRADO['k_torneo'] = 2  (fijado)


---
## FASE 8 - E08_INIT: ¿vale la pena sembrar la población con el Nivel 1?

Rejilla: `tipo_init in ['A', 'B', 'C']`, con `frac_semilla in [0.25, 0.5, 0.75]`
(solo relevante para `C`; las combinaciones redundantes de A y B con distinto
`frac_semilla` se descartan automáticamente). Se mantienen $p_m=0.2$,
$p_c=0.6$, $N=150$, $G=200$ ya fijados.

- **A** - población inicial 100% aleatoria (la usada hasta ahora).
- **B** - población sembrada con el pool de soluciones del Nivel 1 (requiere
  resolver un MILP simplificado con PuLP/CBC).
- **C** - mixta: una fracción `frac_semilla` de la población viene del Nivel 1,
  el resto es aleatoria.

**Objetivo:** decidir si el coste adicional de resolver el Nivel 1 (tiempo de
CBC + complejidad del pipeline) se traduce en una mejora real de factibilidad
o LCOH frente a la inicialización puramente aleatoria.

**ESTE EXPERIMENTO DEBERIAMOS REVISARLO**

In [ ]:
# =====================================================================
# E08_INIT - EJECUCION
# =====================================================================
r_E08 = calibracion.ejecutar_calibracion(
    exp_id="E08_INIT", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E08_INIT", res=r_E08["resumen"], runs=r_E08["runs"],
    conv=pd.read_csv(r_E08["ruta_convergencia"]) if os.path.isfile(r_E08["ruta_convergencia"]) else None,
    cols_rejilla=r_E08["cols_rejilla"], nombre_exp=r_E08["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E08_INIT", runs=r_E08["runs"], res=r_E08["resumen"],
    ranking=r_E08.get("ranking"), nombre_exp=r_E08["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E08["resumen"]["instancia"].tolist())):
    sub = r_E08["resumen"][r_E08["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E08["runs"].loc[r_E08["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E08["runs"].loc[r_E08["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E08_INIT", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E08["cols_rejilla"], nombre_exp=r_E08["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

E08_INIT  |  Estudio de las estrategias de inicializacion de la poblacion
  Instancias      : small, medium, large
  Configuraciones : 5  (init=A_frac=0.25, init=B_frac=0.25, init=C_frac=0.25, init=C_frac=0.5, init=C_frac=0.75)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 300
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=2, init=B

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   init=A_frac=0.25             fact=20/20  LCOH medio=4.6016          [  6.7%]
   init=B_frac=0.25             fact=20/20  LCOH medio=4.6046          [ 13.3%]
   init=C_frac=0.25             fact=20/20  LCOH medio=4.6046          [ 20.0%]
   init=C_frac=0.5              fact=20/20  LCOH medio=4.6046          [ 26.7%]
   init=C_frac=0.75             fact=20/20  LCOH medio=4.6038          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   init=A_frac=0.25             fact=20/20  LCOH m

In [ ]:
ganadoras_E08 = resumen_fase("E08_INIT", r_E08)

VALOR_INIT_ELEGIDO = "B"  # <-- EDITA ESTO: "A", "B" o "C"
VALOR_FRAC_ELEGIDO = None   # <-- EDITA SOLO si se elige "C"

if VALOR_INIT_ELEGIDO is not None:
    C.CALIBRADO["tipo_init"] = VALOR_INIT_ELEGIDO
    print(f"CALIBRADO['tipo_init'] = {VALOR_INIT_ELEGIDO}  (fijado)")
    if VALOR_INIT_ELEGIDO == "C" and VALOR_FRAC_ELEGIDO is not None:
        C.CALIBRADO["frac_semilla"] = VALOR_FRAC_ELEGIDO
        print(f"CALIBRADO['frac_semilla'] = {VALOR_FRAC_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_INIT_ELEGIDO.")

print()
if VALOR_INIT_ELEGIDO == "A":
    print("-> 'A' es suficiente: puedes OMITIR E08b_POOL y E10_MREF "
          "(solo aplican a B/C) y pasar directamente a E09_PENAL.")
elif VALOR_INIT_ELEGIDO in ("B", "C"):
    print("-> Conviene continuar con E08b_POOL (y E10_MREF) para afinar "
          "como se construye el pool del Nivel 1.")

RESUMEN DE E08_INIT
Tiempo total real: 4751.2 s (79.2 min)  |  Tiempo medio/ejecucion: 15.84 s

  [small   ] ganador: init=A_frac=0.25             params={'tipo_init': 'A', 'frac_semilla': np.float64(0.25)}
             LCOH medio = 4.6016  std = 0.0088  factibilidad = 1.00  t medio = 8.41s
  [medium  ] ganador: init=C_frac=0.75             params={'tipo_init': 'C', 'frac_semilla': np.float64(0.75)}
             LCOH medio = 4.3549  std = 0.0370  factibilidad = 1.00  t medio = 15.27s
  [large   ] ganador: init=C_frac=0.5              params={'tipo_init': 'C', 'frac_semilla': np.float64(0.5)}
             LCOH medio = 4.4373  std = 0.0345  factibilidad = 1.00  t medio = 23.63s

  Ranking agregado (mas veces mejor entre instancias):
    init=C_frac=0.75  ->  mejor en 1/3 instancias
CALIBRADO['tipo_init'] = B  (fijado)

-> Conviene continuar con E08b_POOL (y E10_MREF) para afinar como se construye el pool del Nivel 1.


---
## FASE 9 - E09_MREF: modo de referencia del Nivel 1

Rejilla: `m_ref in ['min_efi', 'max_efi', 'min_coste']`, forzando
`tipo_init='B'` (la única inicialización para la que este parámetro tiene
efecto). Se mantienen $p_m=0.2$, $p_c=0.6$, $N=150$, $G=200$ ya fijados.

**Nota:** esta fase resuelve el Nivel 1 con PuLP/CBC para poblar el pool de
semillas, independientemente del resultado de `E08_INIT` (aquí se fuerza
`tipo_init='B'` para poder aislar el efecto de `m_ref`). Si en `E08_INIT`
decides que la inicialización aleatoria (`A`) es suficiente, esta fase queda
como análisis complementario y no condiciona la calibración final.

In [ ]:
# =====================================================================
# E09_MREF - EJECUCION
# =====================================================================
r_E09 = calibracion.ejecutar_calibracion(
    exp_id="E09_MREF", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E09_MREF", res=r_E09["resumen"], runs=r_E09["runs"],
    conv=pd.read_csv(r_E09["ruta_convergencia"]) if os.path.isfile(r_E09["ruta_convergencia"]) else None,
    cols_rejilla=r_E09["cols_rejilla"], nombre_exp=r_E09["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E09_MREF", runs=r_E09["runs"], res=r_E09["resumen"],
    ranking=r_E09.get("ranking"), nombre_exp=r_E09["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E09["resumen"]["instancia"].tolist())):
    sub = r_E09["resumen"][r_E09["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E09["runs"].loc[r_E09["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E09["runs"].loc[r_E09["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E09_MREF", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E09["cols_rejilla"], nombre_exp=r_E09["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

E09_MREF  |  Modo de referencia del Nivel 1
  Instancias      : small, medium, large
  Configuraciones : 3  (mref=min_efi, mref=max_efi, mref=min_coste)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 180
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=2, init=B

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
    [pool] resolviendo Nivel 1 (nogood, n=8) ...
    [pool] 8 soluciones; mejor coste = 10555462.30 EUR
   mref=min_efi                 fact=20/20  LCOH medio=4.6046          [ 11.1%]
    [pool] resolviendo Nivel 1 (nogood, n=8) ...
    [pool] 8 soluciones; mejor coste = 7109981.42 EUR
   mref=max_efi                 fact=0/20  LCOH medio=sin factibles   [ 22.2%]
    [pool] resolviendo Nivel 1 (nogood, n=8) ...
    [pool] 8 soluciones; mejor coste = 7109981.42 EUR
   mref=min_coste               fact=0/20  LCOH medio=sin factibles   [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=

In [ ]:
ganadoras_E09 = resumen_fase("E09_MREF", r_E09)

# --------------------------------------------------------------------
# DECISION: solo relevante si en E08_INIT se eligió tipo_init='B' o 'C'.
# Si se eleigió 'A', puedes dejar esto en None sin ningun efecto practico.
# --------------------------------------------------------------------
VALOR_MREF_ELEGIDO = "min_efi"  # <-- EDITA ESTO: "min_efi", "max_efi" o "min_coste"

if VALOR_MREF_ELEGIDO is not None:
    C.CALIBRADO["m_ref"] = VALOR_MREF_ELEGIDO
    print(f"CALIBRADO['m_ref'] = {VALOR_MREF_ELEGIDO}  (fijado)")
else:
    print("[AVISO] Aun no has fijado VALOR_MREF_ELEGIDO (o no aplica si usas init='A').")

RESUMEN DE E09_MREF
Tiempo total real: 1692.6 s (28.2 min)  |  Tiempo medio/ejecucion: 9.40 s

  [small   ] ganador: mref=min_efi                 params={'m_ref': 'min_efi'}
             LCOH medio = 4.6046  std = 0.0000  factibilidad = 1.00  t medio = 5.00s
  [medium  ] ganador: mref=max_efi                 params={'m_ref': 'max_efi'}
             LCOH medio = 4.3386  std = 0.0251  factibilidad = 1.00  t medio = 9.26s
  [large   ] ganador: mref=max_efi                 params={'m_ref': 'max_efi'}
             LCOH medio = 4.4235  std = 0.0433  factibilidad = 1.00  t medio = 13.65s

  Ranking agregado (mas veces mejor entre instancias):
    mref=max_efi  ->  mejor en 2/3 instancias
CALIBRADO['m_ref'] = min_efi  (fijado)


---
## FASE 10 - E10_PRESUP: ¿más población o más generaciones, a igualdad de coste?

Rejilla: combinaciones `(N, G)` que consumen aproximadamente el mismo
presupuesto de evaluaciones (~8000: `(20,400)`, `(40,200)`, `(80,100)`,
`(160,50)`). Se mantienen $p_m=0.2$, $p_c=0.6$ ya fijados; esta fase NO
sustituye la decisión de $N=150$/$G=200$ ya tomada (Secciones E04/E05/E06),
es un análisis complementario sobre el equilibrio N vs. G a igualdad de
coste computacional.

**Objetivo:** para la memoria, justificar si a presupuesto fijo conviene más
población pequeña con muchas generaciones o población grande con pocas.

In [ ]:
# =====================================================================
# E10_PRESUP - EJECUCION
# =====================================================================
r_E10 = calibracion.ejecutar_calibracion(
    exp_id="E10_PRESUP", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E10_PRESUP", res=r_E10["resumen"], runs=r_E10["runs"],
    conv=pd.read_csv(r_E10["ruta_convergencia"]) if os.path.isfile(r_E10["ruta_convergencia"]) else None,
    cols_rejilla=r_E10["cols_rejilla"], nombre_exp=r_E10["nombre"],
    presupuesto_cte=True, verbose=True,
)
exportar.exportar_todo(
    exp_id="E10_PRESUP", runs=r_E10["runs"], res=r_E10["resumen"],
    ranking=r_E10.get("ranking"), nombre_exp=r_E10["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E10["resumen"]["instancia"].tolist())):
    sub = r_E10["resumen"][r_E10["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E10["runs"].loc[r_E10["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E10["runs"].loc[r_E10["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E10_PRESUP", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E10["cols_rejilla"], nombre_exp=r_E10["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

E10_PRESUP  |  Comparacion a presupuesto computacional constante (N x G = cte)
  Instancias      : small, medium, large
  Configuraciones : 4  (NxG=20x400, NxG=40x200, NxG=80x100, NxG=160x50)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 240
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=2, init=B

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   NxG=20x400                   fact=20/20  LCOH medio=4.6054          [  8.3%]
   NxG=40x200                   fact=20/20  LCOH medio=4.6050          [ 16.7%]
   NxG=80x100                   fact=20/20  LCOH medio=4.6038          [ 25.0%]
   NxG=160x50                   fact=20/20  LCOH medio=4.6046          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   NxG=20x400                   fact=20/20  LCOH medio=4.4130          [ 41.7%]
   NxG=40x200                   fact=20/20  LCOH medio=4.3916          [ 50.0%]
   NxG

In [ ]:
ganadoras_E10 = resumen_fase("E10_PRESUP", r_E10)


RESUMEN DE E10_PRESUP
Tiempo total real: 581.8 s (9.7 min)  |  Tiempo medio/ejecucion: 2.42 s

  [small   ] ganador: NxG=80x100                   params={'presupuesto': '80x100'}
             LCOH medio = 4.6038  std = 0.0034  factibilidad = 1.00  t medio = 1.25s
  [medium  ] ganador: NxG=80x100                   params={'presupuesto': '80x100'}
             LCOH medio = 4.3756  std = 0.0468  factibilidad = 1.00  t medio = 2.41s
  [large   ] ganador: NxG=20x400                   params={'presupuesto': '20x400'}
             LCOH medio = 4.4796  std = 0.0377  factibilidad = 1.00  t medio = 3.71s

  Ranking agregado (mas veces mejor entre instancias):
    NxG=80x100  ->  mejor en 2/3 instancias

Esta fase es un ANALISIS COMPLEMENTARIO: no fija ningun valor en
CALIBRADO. Los valores definitivos de N y G siguen siendo los
fijados en E04_N/E05_G/E06_NG. Usa esta comparacion en la memoria
para discutir el compromiso N vs. G a presupuesto constante.


---
## FASE 11 - E11_PARADA: criterio de parada (generaciones vs. tiempo límite)

Rejilla: `parada in [('generaciones', 100), ('generaciones', 200), ('tiempo', 30.0), ('tiempo', 60.0), ('tiempo', 120.0)]`.
Se mantienen $p_m=0.2$, $p_c=0.6$, $N=150$ ya fijados. Como con `tiempo_max`
fijado el número de generaciones deja de estar controlado, esta fase es
también un análisis complementario, no una calibración que sustituya a
$G=200$.

**Objetivo:** comparar la calidad de solución que se alcanza deteniendo el
GA por número de generaciones frente a detenerlo por tiempo límite, de cara
a decidir el criterio de parada de la versión final del algoritmo.

In [ ]:
# =====================================================================
# E11_PARADA - EJECUCION
# =====================================================================
r_E11 = calibracion.ejecutar_calibracion(
    exp_id="E11_PARADA", instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS,
    guardar_convergencia=True, con_ranking=True, verbose=True,
)
graficas.generar_todas(
    exp_id="E11_PARADA", res=r_E11["resumen"], runs=r_E11["runs"],
    conv=pd.read_csv(r_E11["ruta_convergencia"]) if os.path.isfile(r_E11["ruta_convergencia"]) else None,
    cols_rejilla=r_E11["cols_rejilla"], nombre_exp=r_E11["nombre"], verbose=True,
)
exportar.exportar_todo(
    exp_id="E11_PARADA", runs=r_E11["runs"], res=r_E11["resumen"],
    ranking=r_E11.get("ranking"), nombre_exp=r_E11["nombre"],
    instancias_ids=C.INSTANCIAS, semillas=C.SEMILLAS, verbose=True,
)
for inst_etq in list(dict.fromkeys(r_E11["resumen"]["instancia"].tolist())):
    sub = r_E11["resumen"][r_E11["resumen"]["instancia"] == inst_etq]
    sems = sorted(r_E11["runs"].loc[r_E11["runs"]["instancia"] == inst_etq, "semilla"].unique().tolist())
    t_inst = float(pd.to_numeric(
        r_E11["runs"].loc[r_E11["runs"]["instancia"] == inst_etq, "tiempo_s"],
        errors="coerce").fillna(0.0).sum())
    exportar_calibracion.actualizar_libro_calibracion(
        exp_id="E11_PARADA", instancia_etiqueta=inst_etq, res_instancia=sub,
        cols_rejilla=r_E11["cols_rejilla"], nombre_exp=r_E11["nombre"],
        semillas=sems, tiempo_total_s=t_inst, verbose=True,
    )

E11_PARADA  |  Criterio de parada: generaciones frente a tiempo limite
  Instancias      : small, medium, large
  Configuraciones : 5  (parada=generacionesx100, parada=generacionesx200, parada=tiempox30, parada=tiempox60, parada=tiempox120)
  Semillas        : 20  [101, 202, 303, 404, 505, 606, 707, 808, 909, 1010, 1111, 1212, 1313, 1414, 1515, 1616, 1717, 1818, 1919, 2020]
  Ejecuciones     : 300
  Parametros fijos: N=150, G=200, PC=0.6, PM=0.2, k=2, init=B

[small]  |P|=4 |J|=8 |K|=16 HTotal=22400 kg/dia
   parada=generacionesx100      fact=20/20  LCOH medio=4.6046          [  6.7%]
   parada=generacionesx200      fact=20/20  LCOH medio=4.6046          [ 13.3%]
   parada=tiempox30             fact=20/20  LCOH medio=4.6046          [ 20.0%]
   parada=tiempox60             fact=20/20  LCOH medio=4.6046          [ 26.7%]
   parada=tiempox120            fact=20/20  LCOH medio=4.6046          [ 33.3%]

[medium]  |P|=8 |J|=18 |K|=34 HTotal=38600 kg/dia
   parada=generacionesx100      fact=

In [ ]:
ganadoras_E11 = resumen_fase("E11_PARADA", r_E11)



RESUMEN DE E11_PARADA
Tiempo total real: 13441.0 s (224.0 min)  |  Tiempo medio/ejecucion: 44.80 s

  [small   ] ganador: parada=generacionesx100      params={'parada': 'generacionesx100'}
             LCOH medio = 4.6046  std = 0.0000  factibilidad = 1.00  t medio = 2.48s
  [medium  ] ganador: parada=tiempox120            params={'parada': 'tiempox120'}
             LCOH medio = 4.3618  std = 0.0511  factibilidad = 1.00  t medio = 120.02s
  [large   ] ganador: parada=tiempox120            params={'parada': 'tiempox120'}
             LCOH medio = 4.3923  std = 0.0440  factibilidad = 1.00  t medio = 120.04s

  Ranking agregado (mas veces mejor entre instancias):
    parada=tiempox120  ->  mejor en 2/3 instancias

Esta fase es un ANALISIS COMPLEMENTARIO: no fija ningun valor en
CALIBRADO. El criterio de parada final (por generaciones, G=200)
sigue siendo el usado en el resto de fases. Usa esta comparacion
en la memoria para discutir generaciones vs. tiempo como criterio
de parada del alg